# 03 - Baseline Model
### Week 4: 
- Train a Linear Regression as the first model
- Evaluate using R^2 on the test set
- Record baseline results.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_percentage_error

## Linear Regression

Load + prep data:

In [2]:
train_df = pd.read_csv("data/train_final.csv")
test_df = pd.read_csv("data/test_final.csv")

drop_from_features = ['ClosePrice', 'ClosePrice_log', 'CloseDate', 'CloseYearMonth']

X_train = train_df.drop(columns=[c for c in drop_from_features if c in train_df.columns])
X_test = test_df.drop(columns=[c for c in drop_from_features if c in test_df.columns])

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

y_train = train_df['ClosePrice_log']
y_test = test_df['ClosePrice_log']


In [3]:
# NaN check:
print('X_test NaNs:', X_test.isna().sum().sum())
print('Columns with NaNs in X_test:', X_test.columns[X_test.isna().any()].tolist())

X_test NaNs: 0
Columns with NaNs in X_test: []


Fit model:

In [4]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

### Predict + metrics:

np.exp() is the reverse of np.log() — it converts the log-prediction back into an actual dollar amount before you calculate R²/MAPE/MdAPE against real prices.

In [5]:
pred_log = model.predict(X_test)
pred_price = np.exp(pred_log)
actual_price = np.exp(y_test)

r2 = r2_score(y_test, pred_log)
mape = mean_absolute_percentage_error(actual_price, pred_price)
mdape = np.median(np.abs((actual_price - pred_price) / actual_price))

print(f"R²: {r2:.4f}")
print(f"MAPE: {mape:.4f}")
print(f"MdAPE: {mdape:.4f}")

R²: -1.7771
MAPE: 89308566627551132263185578354521029652151722311680.0000
MdAPE: 0.1698


# 03 - Baseline Model — Summary

**What this notebook does:**
1. Loads `train_final.csv` and `test_final.csv` from `02_preprocessing.ipynb`
2. Drops non-feature columns (`ClosePrice`, `ClosePrice_log`, `CloseDate`, `CloseYearMonth`)
3. Trains a Linear Regression model on log-transformed `ClosePrice`
4. Evaluates on the held-out test month (2026-05)

**Results:**
- R²: 0.7716
- MAPE: 0.2738
- MdAPE: 0.1691

**Takeaway:** Explains ~77% of price variance. MAPE is notably higher than MdAPE, meaning a handful of large errors (likely luxury/high-end properties) are pulling the average up — a sign Linear Regression struggles with nonlinear pricing patterns. This is the baseline the tree-based models in `04_model_comparison.ipynb` need to beat.